In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import mixed_precision

mixed_precision.set_global_policy("float32")

In [2]:
DATA_ROOT = "dataset"
IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS_HEAD = 8
EPOCHS_FINE = 10
VAL_SPLIT = 0.15
SEED = 1337


In [3]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_ROOT,
    validation_split=VAL_SPLIT,
    subset="training",
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="categorical",
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_ROOT,
    validation_split=VAL_SPLIT,
    subset="validation",
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="categorical",
)

class_names = train_ds.class_names
num_classes = len(class_names)

Found 9473 files belonging to 4 classes.
Using 8053 files for training.
Found 9473 files belonging to 4 classes.
Using 1420 files for validation.


In [4]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

In [5]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])


In [6]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])


In [7]:
base_model = keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    pooling="avg",
)

base_model.trainable = False  # freeze for transfer learning

In [8]:
inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

x = data_augmentation(inputs)
x = keras.applications.efficientnet.preprocess_input(x)

x = base_model(x, training=False)

x = layers.BatchNormalization()(x)
x = layers.Dense(256, activation="swish")(x)
x = layers.Dropout(0.5)(x)

outputs = layers.Dense(
    num_classes,
    activation="softmax",
    dtype="float32"
)(x)

model = keras.Model(inputs, outputs)



In [9]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=3e-4),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"],
)


In [10]:
callbacks = [
    keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.3),
]

In [11]:
history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_HEAD,
    callbacks=callbacks,
)

Epoch 1/8
504/504 ━━━━━━━━━━━━━━━━━━━━ 924s 2s/step - accuracy: 0.7073 - loss: 0.9965 - val_accuracy: 0.8141 - val_loss: 0.7392 - learning_rate: 3.0000e-04
Epoch 2/8
504/504 ━━━━━━━━━━━━━━━━━━━━ 1097s 2s/step - accuracy: 0.7842 - loss: 0.8005 - val_accuracy: 0.8261 - val_loss: 0.7071 - learning_rate: 3.0000e-04
Epoch 3/8
504/504 ━━━━━━━━━━━━━━━━━━━━ 905s 2s/step - accuracy: 0.8074 - loss: 0.7457 - val_accuracy: 0.8232 - val_loss: 0.7139 - learning_rate: 3.0000e-04
Epoch 4/8
504/504 ━━━━━━━━━━━━━━━━━━━━ 770s 2s/step - accuracy: 0.8194 - loss: 0.7099 - val_accuracy: 0.8303 - val_loss: 0.6869 - learning_rate: 3.0000e-04
Epoch 5/8
504/504 ━━━━━━━━━━━━━━━━━━━━ 766s 2s/step - accuracy: 0.8320 - loss: 0.6852 - val_accuracy: 0.8359 - val_loss: 0.6684 - learning_rate: 3.0000e-04
Epoch 6/8
504/504 ━━━━━━━━━━━━━━━━━━━━ 872s 2s/step - accuracy: 0.8371 - loss: 0.6725 - val_accuracy: 0.8345 - val_loss: 0.6739 - learning_rate: 3.0000e-04
Epoch 7/8
504/504 ━━━━━━━━━━━━━━━━━━━━ 825s 2s/step - accuracy:

In [12]:
# 🔹 Stage 2: Fine-tuning (VERY IMPORTANT for >80%)

base_model.trainable = True

# Freeze most layers, unfreeze top layers only
for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"],
)

history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FINE,
    callbacks=callbacks,
)

Epoch 1/10
504/504 ━━━━━━━━━━━━━━━━━━━━ 806s 2s/step - accuracy: 0.7844 - loss: 0.7822 - val_accuracy: 0.8254 - val_loss: 0.7142 - learning_rate: 1.0000e-05
Epoch 2/10
504/504 ━━━━━━━━━━━━━━━━━━━━ 796s 2s/step - accuracy: 0.8086 - loss: 0.7209 - val_accuracy: 0.8254 - val_loss: 0.6993 - learning_rate: 1.0000e-05
Epoch 3/10
504/504 ━━━━━━━━━━━━━━━━━━━━ 778s 2s/step - accuracy: 0.8127 - loss: 0.7103 - val_accuracy: 0.8296 - val_loss: 0.6979 - learning_rate: 3.0000e-06
Epoch 4/10
504/504 ━━━━━━━━━━━━━━━━━━━━ 792s 2s/step - accuracy: 0.8208 - loss: 0.7049 - val_accuracy: 0.8338 - val_loss: 0.6966 - learning_rate: 3.0000e-06


In [13]:
model.save("skin_cancer_model.keras")

print("✅ Training complete. Model saved.")


✅ Training complete. Model saved.
